# Prompting, structured output, and tools

**Track:** GenAI applications. **Level:** Intermediate. **CPU:** yes. **Keys:** optional.

## Goals

- Run a zero-shot prompt, a few-shot prompt, and a chain-of-thought prompt through one client.
- Validate a JSON object against required keys and repair it when a key is missing.
- Run a tool loop whose only tools are a calculator and a unit converter.

## Sources and license

Original notebook. The prompt patterns are the ones taught in the Hugging Face LLM course and the Full Stack LLM bootcamp. Neither course notebook is copied. Tool use follows the action/observation loop in Yao et al., *ReAct* (2022). The client is `labs/_shared/llm.py`. The default provider is the local fallback so the notebook is reproducible. Set `LAB_LLM=openai` or `LAB_LLM=anthropic` to send the same messages to a hosted model, or `LAB_LLM=huggingface` to sample a local Transformers model. The repair demo forces the local provider, because the first missing field has to be deterministic.

## Concepts

A prompt is an input to a function with an unstable output. The harness around it is the part you can test: the template, the schema, and the tool allow-list. Structured output means parsing JSON and checking keys, then sending the error back as the next message. A tool loop alternates model actions and tool observations and stops on `final` or on a step limit. The local provider is not a language model. It returns scripted strings so the harness can be graded without a key. The Hugging Face provider in this lab is an untrained tiny GPT-2 unless `HF_MODEL` names a checkpoint. Its text is not graded.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'torch', 'transformers'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

import json
import os

from labs._shared.llm import get_llm

provider = os.environ.get("LAB_LLM", "local")
llm = get_llm(provider)
print("provider", llm.name)


## Guided exercise: three prompts, one client

The local provider answers the France question with `Paris`, and with a short derivation when the prompt says `step by step`. A hosted model receives the same strings. The assertions run only for the local provider. A hosted run prints the text and checks that it is non-empty.


In [ ]:
zero = llm.complete([{"role": "user", "content": "What is the capital of France? Answer in one word."}])
few = llm.complete([{
    "role": "user",
    "content": "Italy: Rome\nSpain: Madrid\nWhat is the capital of France?",
}])
cot = llm.complete([{
    "role": "user",
    "content": "What is the capital of France? Reason step by step, then give the answer.",
}])
print("zero-shot:", zero)
print("few-shot:", few)
print("chain-of-thought:", cot)
if llm.name == "local-fallback":
    assert "Paris" in zero and "Paris" in few and "Paris" in cot
    assert "Answer:" in cot
else:
    assert zero and few and cot


## Guided exercise: schema and repair

Required keys are `city` and `country`. The fresh local client omits `country` on the first call when the prompt contains `repair-demo`, then returns both keys. The loop records the missing-key error. That error is the thing under test.


In [ ]:
def complete_json(client, prompt, keys, retries=3):
    messages = [{"role": "user", "content": prompt}]
    errors = []
    for _ in range(retries):
        raw = client.complete(messages, mode="json")
        try:
            obj = json.loads(raw)
        except json.JSONDecodeError as exc:
            errors.append(str(exc))
            messages.append({"role": "assistant", "content": raw})
            messages.append({"role": "user", "content": prompt + " RETURN_JSON"})
            continue
        missing = [key for key in keys if key not in obj]
        if missing:
            errors.append("missing " + ",".join(missing))
            messages.append({"role": "assistant", "content": raw})
            messages.append({
                "role": "user",
                "content": prompt + " RETURN_JSON include " + ",".join(keys),
            })
            continue
        return obj, errors
    raise RuntimeError(errors)

repair_client = get_llm("local")
obj, errors = complete_json(
    repair_client,
    "repair-demo RETURN_JSON Paris is the city. Name the city and the country.",
    ["city", "country"],
)
print(obj, errors)
assert errors and "country" in errors[0]
assert obj == {"city": "Paris", "country": "France"}


## Guided exercise: a tool loop

The calculator accepts arithmetic. The converter accepts a fixed table of units. Anything else raises, and the loop records the error instead of calling `eval` on it. The transcript uses the markers the local provider understands (`calc:`, `convert:`). A hosted model would be given the same JSON action schema in the system message. This cell stays on the local provider so the trace is stable.


In [ ]:
def calculator(expr: str) -> str:
    import re
    if not re.fullmatch(r"[0-9+*/(). -]+", expr):
        raise ValueError("rejected")
    return str(eval(expr, {"__builtins__": {}}, {}))

def convert(spec: dict) -> str:
    factors = {("km", "m"): 1000.0, ("m", "km"): 0.001, ("kg", "g"): 1000.0, ("g", "kg"): 0.001}
    factor = factors[(spec["src"], spec["dst"])]
    value = float(spec["value"]) * factor
    return str(int(value)) if float(value).is_integer() else str(value)

TOOLS = {"calculator": calculator, "convert": convert}

def run_tools(prompt, client, max_steps=4):
    messages = [{"role": "user", "content": prompt + "\nTOOL_LOOP"}]
    trace = []
    for _ in range(max_steps):
        action = json.loads(client.complete(messages, mode="tools"))
        trace.append(action["action"])
        if action["action"] == "final":
            return trace, str(action["input"])
        observation = TOOLS[action["action"]](action["input"])
        messages.append({"role": "assistant", "content": json.dumps(action)})
        messages.append({"role": "user", "content": f"Observation ({action['action']}): {observation}\nTOOL_LOOP"})
    return trace, ""

agent = get_llm("local")
trace, answer = run_tools("calc: (12 + 5) * 3", agent)
print(trace, answer)
assert trace == ["calculator", "final"] and answer == "51"
trace, answer = run_tools("convert: 1 km to m", agent)
print(trace, answer)
assert trace == ["convert", "final"] and answer == "1000"
try:
    calculator("__import__('os')")
    raise AssertionError("sandbox failed")
except ValueError:
    print("sandbox rejected a non-arithmetic expression")


## Your turn

Ask the local client to convert 2 kg to g. The trace should be `convert`, then `final`, and the answer should be 2000.

## Solution


In [ ]:
trace, answer = run_tools("convert: 2 kg to g", get_llm("local"))
print(trace, answer)
assert trace == ["convert", "final"] and answer == "2000"


## Beyond the course

Call the Hugging Face provider. With no `HF_MODEL` set, this builds an untrained tiny GPT-2 and samples eight characters. The string is prefixed with the checkpoint name. Assert that prefix, and do not assert that the sample means anything. Setting `HF_MODEL` to a public instruct model is how the same function calls a real open model. That download is outside CI.


In [ ]:
hf = get_llm("huggingface")
text = hf.complete([{"role": "user", "content": "Say hello in one short sentence."}])
print(text)
assert text.startswith("[untrained-tiny-gpt2]") or text.startswith("[" + hf.checkpoint + "]")
print("LAB OK")


## Research questions

1. The repair loop trusts the second JSON object once the keys exist. What can a model put in `country` that still passes, and that you would refuse to store?
2. The tool allow-list is two functions. Where in `run_tools` would a third tool name returned by a hosted model be stopped?
3. The chain-of-thought string and the one-word string can name the same city. Which one do you want when you are grading the answer, and which one do you want when you are grading the trace?

## Further reading

- Yao et al., *ReAct: Synergizing Reasoning and Acting in Language Models* (2022).
- Hugging Face LLM course, the prompting and agents chapters (Apache-2.0 code). Not copied. https://huggingface.co/learn/llm-course
- Full Stack LLM bootcamp, the prompting and evaluation labs. Not copied.
- OpenAI and Anthropic tool-use guides, for the JSON action shape a hosted model is asked to emit.
